AI

In [31]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

task 1

In [32]:
def robust_file_reader(f_path: Path | str) -> tuple[str, str]: 
    """Gets a path to a file where it checks if it's UTF-8 or Latin-1 and saves data to a tuple."""
    # Path(Path()) fungerer som Path() så det går bra hvis de skriver path 
    f_path = Path(f_path) 
    byte = f_path.read_bytes()  
    
    try: # Setter dataen til variabel
        file_text_content = byte.decode("utf8") 
        encoding_used = "UTF-8"
    except UnicodeDecodeError: # Hvis den får en feil så setter den til hva det andre valge kunne være
        file_text_content = byte.decode("latin-1")
        encoding_used = "Latin-1"

    return file_text_content, encoding_used


file_text, encoding = robust_file_reader("data/2026/library_cafe/POS_01/register_sales.log")



def log_passer(f_read: str) -> list[dict]:
    """Makes a list of dictonary with valid sales line"""
    valid_list = []

    for line in f_read.splitlines():
        if line.startswith("=") or line.startswith("-") or line.startswith("[DEBUG]"):
            continue

        timestamp_match = re.search(r"\[(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\]\s", line)
        register_id_match = re.search(r"\[([A-Z]{3}_\d{2})\]", line)
        item_match = re.search(r"item=([a-zA-Z]+)\s", line)
        price_match = re.search(r"price=(\d+\.\d+)\s", line)
        currency_match = re.search(r"\s([A-Z]{3})\s", line)
        status_match = re.search(r"\(([A-Z]+)\)", line)

        if timestamp_match and register_id_match and item_match and price_match and currency_match and status_match:
            match_dict = {
                "timestamp": timestamp_match.group(1),
                "register_id": register_id_match.group(1), 
                "item": item_match.group(1),
                "price": float(price_match.group(1)), 
                "currency": currency_match.group(1),
                "status": status_match.group(1)
            }

            valid_list.append(match_dict)
        
    return valid_list


FileNotFoundError: [Errno 2] No such file or directory: 'data\\2026\\library_cafe\\POS_01\\register_sales.log'

task 2

In [ ]:
def recursive_directory_search(path_search: str | Path) -> list[Path]:
    """Checks all the branches under the dictonary it gets"""
    path_search = Path(path_search)
    path_list = []

    for item in path_search.iterdir(): # sjekker igjennom directory og alle filene eller directory inne i item
        if item.is_dir():
            path_list.extend(recursive_directory_search(item))
        elif item.is_file() and item.suffix == ".log": # suffix er det samme som .endswith bare for objekter
            path_list.append(item)
            
    return path_list

print(recursive_directory_search(Path("data")))

file_text, encoding = robust_file_reader(Path("data/store_config.json"))
info = json.loads(file_text)


def register_travel_recursive(value: list | dict) -> set[str]:
    """Checks if it got a list or dictonary and cheks if it have id's that is active and returns it"""
    set_id = set()

    if isinstance(value, dict):
        if value.get("active") is True: 
            set_id.add(value["id"])
        for inside in value.values():
            set_id.update(register_travel_recursive(inside))
    elif isinstance(value, list):
        for inside in value:
            set_id.update(register_travel_recursive(inside))

    return set_id

print(register_travel_recursive(info)) # ta bort



[]
{'POS_02', 'POS_03', 'POS_01'}


**task 3** <br>
Functions for cleaning and analysing data using NumPy

In [ ]:
def remove_price_outliers(price_data: list[float] | np.ndarray) -> np.ndarray:
    """
    Removes hardware glitch outliers from price data.

    Converts input price data to a 1D NumPy float array.
    Detects outliers where |x - μ| > 3σ.
    Replaces outliers with the median of valid prices.

    Parameters:
        price_data : array-like
            Input price values.

    Returns:
        np.ndarray
            Cleaned 1D float array.
    """

    # Convert input to a 1D NumPy float array
    prices = np.asarray(price_data, dtype=float).flatten()

    # Handle empty input
    if prices.size == 0:
        return prices

    # Calculate mean and standard deviation
    mean = np.mean(prices)
    std = np.std(prices)

    # Detect outliers: |x - μ| > 3σ
    outliers = np.abs(prices - mean) > 3 * std

    # Find median of valid prices
    valid_prices = prices[~outliers]
    median_value = np.median(valid_prices)

    # Replace outliers with median
    prices[outliers] = median_value

    return prices


def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Computes the element-wise refund ratio.

    Refund Ratio = Refund Count / (Sales Count + 1e-6)

    Parameters:
        sales_count : np.ndarray
        refund_count : np.ndarray

    Returns:
        np.ndarray
            Array of refund ratios.
    """

    epsilon = 1e-6

    # Accept both lists and NumPy arrays
    sales = np.asarray(sales_count, dtype=float)
    refunds = np.asarray(refund_count, dtype=float)

    return refunds / (sales + epsilon)



In [ ]:
# Tests for Task 3

# remove_price_outliers: needs enough normal values for 999 to be > 3σ from the mean
prices = [4.0] * 20 + [999.0]
cleaned = remove_price_outliers(prices)
assert len(cleaned) == len(prices)
assert 999.0 not in cleaned            # outlier removed
assert np.isclose(cleaned[-1], 4.0)    # replaced with the median
assert prices[-1] == 999.0             # original input unchanged
print("remove_price_outliers: OK!")

# Empty input
assert remove_price_outliers([]).size == 0
print("remove_price_outliers (empty): OK!")

# compute_refund_ratio with arrays
ratios = compute_refund_ratio(np.array([100, 50, 0]), np.array([5, 2, 0]))
assert np.allclose(ratios, [0.05, 0.04, 0.0])   # last one: 0/0 gives 0, no crash
print("compute_refund_ratio (arrays): OK!")

# compute_refund_ratio with lists
ratios = compute_refund_ratio([100, 50], [5, 2])
assert np.allclose(ratios, [0.05, 0.04])
print("compute_refund_ratio (lists): OK!")

print("\nAll Task 3 tests passed!")

remove_price_outliers: OK!
remove_price_outliers (empty): OK!
compute_refund_ratio (arrays): OK!
compute_refund_ratio (lists): OK!

All Task 3 tests passed!


**task 4** <br>
Data analysis using Pandas

In [ ]:
def create_transaction_dataframe(
    transactions: list[dict[str, str | float]]
) -> pd.DataFrame:
    """
    Convert a list of transaction dictionaries into a DataFrame.
    Convert timestamp to datetime, set it as index,
    and sort chronologically.
    """

    df = pd.DataFrame(transactions)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()

    return df


def hourly_item_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group by item and resample into 1-hour windows,
    calculating total price (sum).
    """

    return (
        df.groupby("item")
          .resample("1h")["price"]
          .sum()
          .reset_index()
    )


def revenue_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a pivot table displaying total transaction
    revenue grouped by item across status categories.
    """

    pivot = pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0,
    )

    return pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"],
        fill_value=0
    )


def refund_ratio_per_item(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate refund ratio per item using
    compute_refund_ratio from Task 3.
    """

    sales_counts = (
        df[df["status"] == "SUCCESS"]
        .groupby("item")
        .size()
    )

    refund_counts = (
        df[df["status"] == "REFUND"]
        .groupby("item")
        .size()
    )

    summary = pd.DataFrame({
        "sales_count": sales_counts,
        "refund_count": refund_counts,
    }).fillna(0)

    summary["refund_ratio"] = compute_refund_ratio(
        summary["sales_count"],
        summary["refund_count"]
    )

    return summary



In [ ]:
#testcell
# Test data: small, hand-made list of transactions (same format as the Task 1 parser output)
# Deliberately NOT in chronological order, and no CANCELLED transactions
test_transactions = [
    {"timestamp": "2026-10-15 09:15:00", "register_id": "POS_01", "item": "Latte",
     "price": 4.50, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 08:30:00", "register_id": "POS_01", "item": "Latte",
     "price": 4.50, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 08:45:00", "register_id": "POS_02", "item": "Latte",
     "price": 4.50, "currency": "EUR", "status": "REFUND"},
    {"timestamp": "2026-10-15 08:50:00", "register_id": "POS_02", "item": "Tea",
     "price": 3.00, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 08:55:00", "register_id": "POS_02", "item": "Tea",
     "price": 2.00, "currency": "EUR", "status": "SUCCESS"},
]

# Task 4.1: create_transaction_dataframe
test_df = create_transaction_dataframe(test_transactions)
assert isinstance(test_df.index, pd.DatetimeIndex)
assert "timestamp" not in test_df.columns
assert test_df.index.is_monotonic_increasing
assert len(test_df) == 5
print("4.1 create_transaction_dataframe: OK!")

# Task 4.2: hourly_item_revenue
hourly = hourly_item_revenue(test_df)
latte_08 = hourly[(hourly["item"] == "Latte") &
                  (hourly["timestamp"] == pd.Timestamp("2026-10-15 08:00"))]["price"].iloc[0]
tea_08 = hourly[(hourly["item"] == "Tea") &
                (hourly["timestamp"] == pd.Timestamp("2026-10-15 08:00"))]["price"].iloc[0]
assert np.isclose(latte_08, 9.00)   # 4.50 SUCCESS + 4.50 REFUND in the 08:00 window
assert np.isclose(tea_08, 5.00)     # 3.00 + 2.00
assert np.isclose(hourly["price"].sum(), test_df["price"].sum())  # no revenue lost
print("4.2 hourly_item_revenue: OK!")

# Task 4.3: revenue_pivot_table
pivot = revenue_pivot_table(test_df)
assert set(pivot.columns) == {"SUCCESS", "REFUND", "CANCELLED"}  # CANCELLED exists even with no data
assert np.isclose(pivot.loc["Latte", "SUCCESS"], 9.00)
assert np.isclose(pivot.loc["Latte", "REFUND"], 4.50)
assert np.isclose(pivot.loc["Tea", "REFUND"], 0.0)
assert np.isclose(pivot.loc["Tea", "CANCELLED"], 0.0)
print("4.3 revenue_pivot_table: OK!")

# Task 4.4: refund_ratio_per_item
ratios = refund_ratio_per_item(test_df)
assert np.isclose(ratios.loc["Latte", "refund_ratio"], 1 / 2, atol=1e-5)  # 1 refund / 2 sales
assert np.isclose(ratios.loc["Tea", "refund_ratio"], 0.0)                 # 0 refunds
print("4.4 refund_ratio_per_item: OK!")

print("\nAll Task 4 tests passed!")

4.1 create_transaction_dataframe: OK!
4.2 hourly_item_revenue: OK!
4.3 revenue_pivot_table: OK!
4.4 refund_ratio_per_item: OK!

All Task 4 tests passed!


task 5

In [ ]:

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

#1. Recursively find all log files and extract active register IDs.
log_files = recursive_directory_search(DATA_DIR / "2026")

config_text, _ = robust_file_reader(DATA_DIR / "store_config.json")
active_ids = register_travel_recursive(json.loads(config_text))

print(f"Found {len(log_files)} log files")
print(f"Active registers: {sorted(active_ids)}")

#2. Ingest text logs and parse records into a list of dictionaries.
all_transactions = []


for log_file in log_files:
   file_text, _ = robust_file_reader(log_file)
   all_transactions.extend(log_passer(file_text))

print(f"Parsed {len(all_transactions)} valid transactions")

#3. Build the initial Pandas DataFrame
df = create_transaction_dataframe(all_transactions)

#keep only active registers
df = [df["register_id"].isin(active_ids)]

#4. Apply NumPy outlier cleaning to remove price glitches
n_outliers = (df["price"] == 999.0).sum()

df["price"] = remove_price_outliers(df["price"])

print(
   f"Replaced {n_outliers} outliers, "
   f"mac price is now {df['price'].max():.2f}"
)

#5. Apply currency correction for NOK transactions
df.loc[df["currency"] == "NOK", "price"] *= 10.0

#6. Generate reports
hourly = hourly_item_revenue(df)

pivot = revenue_pivot_table(df)

refund_ratios = refund_ratio_per_item(df)

#7. Save output files
df.to_csv(
   OUTPUT_DIR / "cleaned_transactions.csv"
)

hourly.to_json(
   OUTPUT_DIR / "hourly_revenue.json",
   orient="records",
   date_format="iso",
   indent=2
)

pivot.to_csv(
   OUTPUT_DIR / "status_pivot_table.csv"
)

refund_ratios.to_csv(
   OUTPUT_DIR / "item_refund_ratios.csv"
)

print(f"All output files saved to {OUTPUT_DIR}")

display(pivot)
display(refund_ratios)



Found 0 log files
Active registers: ['POS_01', 'POS_02', 'POS_03']
Parsed 0 valid transactions


KeyError: 'timestamp'